# Visão da Analytical Base Table (ABT)

Este notebook valida `Dados/abt.csv`: granularidade, quantidade de features, cobertura dos históricos, qualidade dos dados e relação das variáveis com o `TARGET`.

As regras e helpers ficam centralizados em `DataPipeline/pipeline_functions.py`; as células abaixo apenas chamam essas funções e exibem os resultados.


### Bloco 1 — Preparação do ambiente
**O que este bloco faz:** localiza a raiz do projeto, importa Pandas/NumPy/Matplotlib e habilita a camada de storage usada pelo pipeline.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from DataPipeline import config
from DataPipeline import pipeline_functions as pf


### Bloco 2 — Carregamento e validação da granularidade
**O que este bloco faz:** lê a ABT, mostra seu shape, verifica se existe uma única linha por `SK_ID_CURR` e calcula a taxa de inadimplência preservada.

In [ ]:
abt = pf.load_abt()
overview = pf.dataset_overview(abt)

print(f"ABT: {overview['rows']:,} linhas x {overview['columns']:,} colunas")
print(f"IDs únicos: {overview['unique_ids']:,}")
print(f"Duplicidades de ID: {overview['duplicate_ids']:,}")
print(f"Taxa de default: {overview['target_rate']:.2%}")
print(f"Features de entrada antes do encoding: {overview['columns'] - 2:,}")


### Bloco 3 — Composição da ABT por origem
**O que este bloco faz:** separa as colunas em identificador/alvo, Application, Bureau, Previous e razões financeiras; depois conta quantas features existem em cada bloco.

In [ ]:
block_table = pf.feature_block_table(abt)
display(block_table)
pf.plot_feature_block_counts(block_table)


### Bloco 4 — Lista das features históricas
**O que este bloco faz:** exibe exatamente quais colunas `BUREAU_*` e `PREV_*` chegaram à ABT e conta as features de cada fonte.

In [ ]:
history = pf.historical_feature_lists(abt)
print(f"BUREAU_*: {len(history['Bureau'])} features")
print(history["Bureau"])
print()
print(f"PREV_*: {len(history['Previous'])} features")
print(history["Previous"])


### Bloco 5 — Cobertura dos históricos
**O que este bloco faz:** calcula a proporção de clientes que possuem pelo menos um crédito no bureau e pelo menos uma solicitação anterior.

In [ ]:
display(pf.history_coverage(abt).style.format("{:.2%}"))


### Bloco 6 — Qualidade da ABT
**O que este bloco faz:** cria uma tabela com tipo de dado, percentual de nulos e cardinalidade de cada coluna para verificar se a ABT está pronta para o Pipeline de modelagem.

In [ ]:
display(pf.quality_summary(abt).head(35))


### Bloco 7 — Histórico externo por TARGET
**O que este bloco faz:** compara médias de algumas features do bureau entre adimplentes e inadimplentes para observar diferenças de dívida, atraso e quantidade de créditos.

In [ ]:
display(pf.target_group_summary(abt, config.ABT_BUREAU_CHECK_COLUMNS))


### Bloco 8 — Histórico interno por TARGET
**O que este bloco faz:** compara quantidade de pedidos anteriores, aprovações, recusas e taxas históricas entre adimplentes e inadimplentes.

In [ ]:
display(pf.target_group_summary(abt, config.ABT_PREVIOUS_CHECK_COLUMNS))


### Bloco 9 — Correlação numérica com o TARGET
**O que este bloco faz:** calcula a correlação linear das variáveis numéricas com `TARGET`, ordena pelo valor absoluto e exibe as associações mais fortes.

In [ ]:
correlations = pf.numeric_target_correlations(abt)
display(correlations.head(30))
pf.plot_target_correlations(correlations)


### Bloco 10 — Distribuição das razões financeiras
**O que este bloco faz:** compara as três razões financeiras entre as classes usando boxplots, limitando a visualização ao percentil 99 para reduzir o efeito de outliers extremos no gráfico.

In [ ]:
pf.plot_ratio_boxplots(abt)


### Bloco 11 — Checklist de prontidão
**O que este bloco faz:** não altera a base. Registra as condições esperadas antes do treinamento.

- Uma linha por `SK_ID_CURR`.
- `TARGET` preservado.
- 15 features `BUREAU_*` quando todas as colunas necessárias estão disponíveis.
- 13 features `PREV_*` quando todas as colunas necessárias estão disponíveis.
- Histórico inexistente preenchido com zero.
- Categóricas e nulos restantes preservados para tratamento dentro do Pipeline do Scikit-Learn.
- ABT pronta para split treino/holdout.